In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
PROJECT_DIR = Path("/content/project/DEPI-GRADUATION-PROJECT-task-2")
DATA_DIR = PROJECT_DIR / "data" / "processed" / "price_review_task4"
print("DATA_DIR:", DATA_DIR)
print("Files:", [f.name for f in DATA_DIR.iterdir()])

DATA_DIR: /content/project/DEPI-GRADUATION-PROJECT-task-2/data/processed/price_review_task4
Files: ['review_Sheet_final_100_rows.xlsx', 'train_scored_4_1.csv', 'val_scored_4_1.csv']


In [2]:
review_sheet = pd.read_excel(
    DATA_DIR / "review_Sheet_final_100_rows.xlsx"
)
print("Shape:", review_sheet.shape)
display(review_sheet.head())

Shape: (100, 19)


,ad_id,ad_url,side,title,brand,model_key,price,ref_median_price,price_ratio,mad_z,price_review_reason,price_context,decision,fixed_price,fixed_model_key,error_type,notes,reason_count,dataset_action
0,208989958,/ad/%D8%A2%D8%A8%D9%84-%D8%A2%D9%8A%D9%81%D9%8...,lowest,آبل - آيفون 17 برو ماكس,apple,apple iphone 17 pro max,1000,83000,0.01,-29.81,price_placeholder;ratio_low;mad_low,NaN,drop,NaN,NaN,placeholder_price,DROP: Exclude this ad from the cleaned dataset...,3,DELETE
1,208954191,/ad/%D8%A2%D8%A8%D9%84-%D8%A2%D9%8A%D9%81%D9%8...,lowest,آبل - آيفون 17 برو ماكس,apple,apple iphone 17 pro max,1000,83000,0.01,-29.81,price_placeholder;ratio_low;mad_low,NaN,drop,NaN,NaN,placeholder_price,DROP: Exclude this ad from the cleaned dataset...,3,DELETE
2,208983975,/ad/%D8%A2%D8%A8%D9%84-%D8%A2%D9%8A%D9%81%D9%8...,lowest,آبل - آيفون 17 برو,apple,apple iphone 17 pro,1000,74000,0.01,-20.95,price_placeholder;ratio_low;mad_low,NaN,drop,NaN,NaN,placeholder_price,DROP: Exclude this ad from the cleaned dataset...,3,DELETE
3,209009851,/ad/%D8%B3%D8%A7%D9%85%D8%B3%D9%88%D9%86%D8%AC...,lowest,سامسونج جلاكسي اس 26 الترا,samsung,samsung s26 ultra,1000,67000,0.01,-28.36,price_placeholder;ratio_low;mad_low,NaN,drop,NaN,NaN,placeholder_price,DROP: Exclude this ad from the cleaned dataset...,3,DELETE
4,208942594,/ad/%D8%A2%D8%A8%D9%84-%D8%A2%D9%8A%D9%81%D9%8...,lowest,آبل - آيفون 17 برو ماكس,apple,apple iphone 17 pro max,1500,83000,0.02,-27.07,ratio_low;mad_low,NaN,drop,NaN,NaN,unresolved,DROP: Exclude this ad from the cleaned dataset...,2,DELETE


In [3]:
print("Decision distribution:")
print(review_sheet["decision"].value_counts(dropna=False))
print("\nDataset action distribution:")
print(review_sheet["dataset_action"].value_counts(dropna=False))

Decision distribution:
decision
drop    56
keep    43
fix      1
Name: count, dtype: int64

Dataset action distribution:
dataset_action
DELETE                56
KEEP                  43
FIX BEFORE KEEPING     1
Name: count, dtype: int64


In [4]:
fix_rows = review_sheet[review_sheet["decision"] == "fix"]
display(fix_rows)

,ad_id,ad_url,side,title,brand,model_key,price,ref_median_price,price_ratio,mad_z,price_review_reason,price_context,decision,fixed_price,fixed_model_key,error_type,notes,reason_count,dataset_action
65,208823727,/ad/%D8%B4%D8%A7%D9%88%D9%85%D9%8A-15-t-%D9%84...,highest,شاومي 15 t للبدل اقرا الوصف,xiaomi,xiaomi 15,22000,11000,2.0,1.36,NaN,exchange,fix,NaN,xiaomi 15t,wrong_model,FIX: Correct the identified field before inclu...,0,FIX BEFORE KEEPING


In [7]:
train_df = pd.read_csv(DATA_DIR / "train_scored_4_1.csv")
print("Train shape:", train_df.shape)

Train shape: (5165, 64)


In [8]:
ad_id = 208823727
train_df[train_df["ad_id"] == ad_id]

,title,price,location,ad_url,ad_id,run_id,brand,storage_gb,ram_gb,storage_src,...,ref_n,ref_med_log,ref_mad_log,ref_level,ref_median_price,price_ratio,mad_z,price_review_reason,price_flag,price_context
4574,شاومي 15 t للبدل اقرا الوصف,22000.0,العوايد، الإسكندرية,/ad/%D8%B4%D8%A7%D9%88%D9%85%D9%8A-15-t-%D9%84...,208823727,legacy,xiaomi,256.0,12.0,NaN,...,7.0,9.305651,0.342945,model_key,11000.0,2.0,1.363274,NaN,False,exchange


In [9]:
train_df.loc[
    train_df["ad_id"] == 208823727,
    ["ad_id", "title", "brand", "model_key", "price"]
]

,ad_id,title,brand,model_key,price
4574,208823727,شاومي 15 t للبدل اقرا الوصف,xiaomi,xiaomi 15,22000.0


In [10]:
# Create a copy of the original train dataset
clean_train_df = train_df.copy()
drop_ids = review_sheet.loc[
    review_sheet["decision"] == "drop",
    "ad_id"
]
clean_train_df = clean_train_df[
    ~clean_train_df["ad_id"].isin(drop_ids)
].copy()
fix_rows = review_sheet[
    review_sheet["decision"] == "fix"
]
for _, row in fix_rows.iterrows():

    ad_id = row["ad_id"]

    if pd.notna(row["fixed_model_key"]):

        clean_train_df.loc[
            clean_train_df["ad_id"] == ad_id,
            "model_key"
        ] = row["fixed_model_key"]

print("Original train shape:", train_df.shape)
print("Clean train shape:", clean_train_df.shape)

print("Removed ads:", len(drop_ids))
print("Fixed ads:", len(fix_rows))

Original train shape: (5165, 64)
Clean train shape: (5109, 64)
Removed ads: 56
Fixed ads: 1


In [11]:
clean_train_path = DATA_DIR / "train_clean_4_3.csv"
clean_train_df.to_csv(clean_train_path, index=False)
print("Saved successfully:", clean_train_path)

Saved successfully: /content/project/DEPI-GRADUATION-PROJECT-task-2/data/processed/price_review_task4/train_clean_4_3.csv


In [12]:
clean_train_df.loc[
    clean_train_df["ad_id"] == 208823727,
    ["ad_id", "title", "brand", "model_key", "price"]
]

,ad_id,title,brand,model_key,price
4574,208823727,شاومي 15 t للبدل اقرا الوصف,xiaomi,xiaomi 15t,22000.0


In [13]:
clean_train_df["ad_id"].isin(drop_ids).sum()

np.int64(0)

In [14]:
print("File exists:", clean_train_path.exists())
print("File size:", clean_train_path.stat().st_size, "bytes")

File exists: True
File size: 3256431 bytes
